<a href="https://colab.research.google.com/github/omsawarkar/omsawarkar/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omsawarkar/omsawarkar/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

I will use the transparent Week 4 peer-gap baseline to produce an editor-review queue. The Week 6 audit did not establish a consistent advantage for logistic regression, so additional model complexity is not justified for this playbook.

The score uses only 1–20 March 2026 observations. It equals impressions multiplied by the positive difference between the item’s CTR and its position-and-volume peer benchmark, calculated as max(peer median CTR − item CTR, 0).

This is a historical demonstration, not a current operational queue. The score is a descriptive priority index, not expected recoverable clicks or proof that updating a page will help.

| Evidence category | Reason code | Suggested action |
|---|---|---|
| Enough observations and CTR below its peer benchmark | BELOW_PEER_CTR | Review search appearance, query intent, and measurement quality before considering a change. |
| Enough observations but no positive peer gap | NO_CTR_GAP | Monitor; this rule provides no CTR-based reason to prioritize an edit. |
| Insufficient observations, volume, position information, or peer support | INSUFFICIENT_DATA | Gather or verify data before assigning review priority. |

These are rule-defined evidence categories, not learned content archetypes. Within the review group, higher scores come first, with deterministic tie-breaking.

The earlier decay and refresh question remains a motivation, not a measured result of this rule. The daily table does not provide update history, so the queue cannot establish staleness or prescribe a refresh. An editor must inspect the actual page before choosing an intervention.

In [4]:
%pip -q install huggingface_hub duckdb

import pandas as pd
import numpy as np
import duckdb
from pathlib import Path

from google.colab import userdata
from huggingface_hub import hf_hub_download

token = userdata.get("HF_TOKEN").strip()

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename=(
        "fact_content_daily_performance/"
        "month=2026-03/data_0.parquet"
    ),
    repo_type="dataset",
    token=token
)

con = duckdb.connect()
con.read_parquet(
    march_path,
    hive_partitioning=True
).create_view("march_raw")

print("March dataset ready.")

display(con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM march_raw
""").df())


March dataset ready.


,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [5]:
# Build the early-window content table.
pages = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS observed_days,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS ctr,
        SUM(ga4_sessions) AS sessions,
        SUM(ga4_engaged_sessions) * 1.0
            / NULLIF(SUM(ga4_sessions), 0) AS engagement_rate,

        SUM(
            CASE WHEN gsc_avg_position > 0
                 THEN gsc_avg_position * gsc_impressions END
        ) / NULLIF(
            SUM(
                CASE WHEN gsc_avg_position > 0
                     THEN gsc_impressions END
            ), 0
        ) AS avg_position

    FROM march_raw
    WHERE report_date BETWEEN DATE '2026-03-01'
                          AND DATE '2026-03-20'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING COUNT(*) >= 10
       AND SUM(gsc_impressions) > 0
""").df()

pages["position_bucket"] = pd.cut(
    pages["avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1–3", ">3–10", ">10–20", ">20"]
)

pages["volume_bucket"] = pd.cut(
    pages["impressions"],
    bins=[99, 499, 1999, float("inf")],
    labels=["100–499", "500–1,999", "2,000+"]
)

peer_stats = (
    pages.loc[
        (pages["impressions"] >= 100)
        & pages["position_bucket"].notna()
    ]
    .groupby(
        ["position_bucket", "volume_bucket"],
        observed=True
    )
    .agg(
        peer_count=("ctr", "size"),
        peer_median_ctr=("ctr", "median")
    )
    .reset_index()
)

queue = pages.merge(
    peer_stats,
    on=["position_bucket", "volume_bucket"],
    how="left",
    validate="many_to_one"
)

eligible = (
    (queue["observed_days"] >= 10)
    & (queue["impressions"] >= 100)
    & queue["position_bucket"].notna()
    & (queue["peer_count"] >= 20)
    & queue["ctr"].notna()
    & queue["peer_median_ctr"].notna()
)

queue["ctr_gap"] = (
    queue["peer_median_ctr"] - queue["ctr"]
).clip(lower=0)

queue["score"] = np.where(
    eligible,
    queue["impressions"] * queue["ctr_gap"],
    0.0
)

queue["reason_code"] = np.select(
    [~eligible, queue["score"] > 0],
    ["INSUFFICIENT_DATA", "BELOW_PEER_CTR"],
    default="NO_CTR_GAP"
)

queue["action"] = queue["reason_code"].map({
    "BELOW_PEER_CTR": "REVIEW_SEARCH_APPEARANCE_AND_INTENT",
    "NO_CTR_GAP": "MONITOR",
    "INSUFFICIENT_DATA": "COLLECT_MORE_DATA"
})

queue["review_status"] = "NOT_REVIEWED"
queue["confidence_note"] = (
    "Descriptive evidence only; intervention benefit unverified"
)

queue = queue.sort_values(
    ["score", "impressions", "client_hash_id", "content_hash_id"],
    ascending=[False, False, True, True]
).reset_index(drop=True)

queue.insert(0, "rank", np.arange(1, len(queue) + 1))

assert not queue.duplicated(
    ["client_hash_id", "content_hash_id"]
).any()
assert queue["score"].notna().all()
assert (queue["score"] >= 0).all()

print("Queue items:", len(queue))
print("Positive-score items:", (queue["score"] > 0).sum())

print("Items in each action category:")
display(queue["action"].value_counts().to_frame("item_count"))

print("Top 10 historical review candidates:")
display(queue[[
    "rank", "score", "impressions", "clicks",
    "position_bucket", "volume_bucket",
    "reason_code", "action", "review_status"
]].head(10).round(3))

Queue items: 6607
Positive-score items: 3280
Items in each action category:


,item_count
action,
MONITOR,3285
REVIEW_SEARCH_APPEARANCE_AND_INTENT,3280
COLLECT_MORE_DATA,42


Top 10 historical review candidates:


,rank,score,impressions,clicks,position_bucket,volume_bucket,reason_code,action,review_status
0,1,407.942,176168.0,466.0,>10–20,"2,000+",BELOW_PEER_CTR,REVIEW_SEARCH_APPEARANCE_AND_INTENT,NOT_REVIEWED
1,2,344.424,75879.0,32.0,>10–20,"2,000+",BELOW_PEER_CTR,REVIEW_SEARCH_APPEARANCE_AND_INTENT,NOT_REVIEWED
2,3,292.604,85591.0,132.0,>10–20,"2,000+",BELOW_PEER_CTR,REVIEW_SEARCH_APPEARANCE_AND_INTENT,NOT_REVIEWED
3,4,275.442,51074.0,37.0,>3–10,"2,000+",BELOW_PEER_CTR,REVIEW_SEARCH_APPEARANCE_AND_INTENT,NOT_REVIEWED
4,5,249.471,42088.0,8.0,>3–10,"2,000+",BELOW_PEER_CTR,REVIEW_SEARCH_APPEARANCE_AND_INTENT,NOT_REVIEWED
5,6,216.742,41315.0,36.0,>3–10,"2,000+",BELOW_PEER_CTR,REVIEW_SEARCH_APPEARANCE_AND_INTENT,NOT_REVIEWED
6,7,197.102,53307.0,129.0,>3–10,"2,000+",BELOW_PEER_CTR,REVIEW_SEARCH_APPEARANCE_AND_INTENT,NOT_REVIEWED
7,8,188.124,48243.0,107.0,>3–10,"2,000+",BELOW_PEER_CTR,REVIEW_SEARCH_APPEARANCE_AND_INTENT,NOT_REVIEWED
8,9,181.575,47663.0,110.0,>3–10,"2,000+",BELOW_PEER_CTR,REVIEW_SEARCH_APPEARANCE_AND_INTENT,NOT_REVIEWED
9,10,181.381,36679.0,43.0,>3–10,"2,000+",BELOW_PEER_CTR,REVIEW_SEARCH_APPEARANCE_AND_INTENT,NOT_REVIEWED


## 2. Intended use and limits

### Intended user and purpose

This playbook is for an editor or SEO analyst choosing which pages to investigate first when review time is limited. It provides a historical demonstration using 1–20 March 2026 data, not a current operational recommendation.

The queue contains 6,607 client–content pairs: 3,280 review candidates, 3,285 monitoring items, and 42 items requiring more evidence. All are initially marked NOT_REVIEWED.

### Coverage and evidence

The initial population requires at least 10 days with both GSC and GA4 available and positive search impressions. Items excluded by these initial filters are absent from the queue, rather than classified as monitoring items.

Peer comparisons use broad position and impression-volume groups. They mix clients and do not control for query intent, device, country, or search-result layout. Observation-day coverage also differs between items.

### Decay and refresh interpretation

Low relative CTR can justify investigation of search appearance or intent alignment. It does not establish content decay or staleness. This rule has no update-history input, so an editor must inspect factual freshness and content usefulness before considering a refresh.

### Cost and value

As a proposed pilot, review the first 10 candidates with an initial triage budget of 15 minutes per item: 150 minutes in total. These are planning assumptions, not measured costs. Record actual review time and stop or escalate cases needing deeper investigation.

Track how many reviews identify a specific, actionable issue and how much time each takes. Compare review yield and time with the existing manual process. Do not convert the score into revenue or promised additional clicks.

### Limits

Historical proxy validation did not establish human-confirmed review quality or content-update benefit. No automatic editing, pruning, or guaranteed traffic improvement is justified. A zero score means this rule found no positive CTR gap; it does not mean the page is healthy or unimportant.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

### Human-review checklist

For each candidate, the editor or analyst should:

1. Verify measurement availability, date coverage, impressions, clicks, and position. Investigate suspicious counts before recommending changes.
2. Check whether the peer comparison is appropriate for the client, query intent, device, country, and search-result context.
3. Inspect the actual page and its search appearance through authorized access. Check whether the title and description accurately represent the content and whether the page answers its intended question.
4. Look for outdated facts, missing information, duplication, or relevance issues. Do not infer these problems solely from the score.
5. Record the finding, supporting evidence, estimated effort, reviewer, and decision.
6. Choose one outcome: APPROVE_SPECIFIC_CHANGE, INVESTIGATE_FURTHER, MONITOR, or REJECT_RECOMMENDATION.
7. Obtain the content owner’s approval before implementing a change. Record its date and arrange follow-up measurement.

No actual pages have been inspected in this notebook. Queue entries remain NOT_REVIEWED until that work occurs.

### What must not be automated

- Publishing, rewriting, deleting, pruning, redirecting, or changing indexing settings based only on the score.
- Treating a low score as proof that a page is healthy.
- Inferring client identities from pseudonyms or exporting private queries or content.
- Claiming that an edit caused a traffic change without an appropriate evaluation design.
- Sending recommendations to clients without an authorized person’s review.

### Stop and escalate

Pause recommendations if tracking is unreliable, evidence is insufficient, the peer group is unsuitable, or the proposed change affects sensitive or business-critical content. The reviewer should explain the uncertainty and seek the content owner’s judgment.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

### Proposed monitoring schedule

This notebook demonstrates a historical workflow. Before operational use, an analyst must obtain a new authorized data window, verify reporting completeness, and review the rule’s suitability.

During a pilot, check data quality before each weekly queue refresh. Use consistent window lengths and availability rules. Freeze each queue’s configuration and save a run receipt.

### Data-quality triggers

Pause scoring if required columns disappear, identifiers or dates are missing, duplicate keys appear, or calculations produce invalid values. Treat unusual CTR or engagement rates as investigation triggers rather than silently clipping them.

Review any substantial change in availability coverage, eligible-item counts, position distribution, or peer-group sizes. As a provisional alert, investigate a greater than 20% relative change in eligible-item count compared with the preceding equivalent window. This threshold is a planning choice, not a validated statistical limit.

### Review-quality triggers

For each batch of 10 reviewed candidates, record confirmed actionable issues, rejected recommendations, unresolved cases, and review time. Define the checklist before reviewing. Report confirmed actions out of all 10, with unresolved cases disclosed.

As a provisional pilot goal, seek at least 7 actionable issues among 10 candidates. If two consecutive batches fall below that goal, pause expansion and inspect the failures. This is not an achieved result, and small batches have substantial uncertainty.

### Follow-up after approved changes

Check search and engagement measurements approximately 30 and 60 days after implementation, allowing for reporting delays. Record other changes, seasonality, and comparable untreated pages where available. A before-and-after difference alone does not establish causation.

### Recalibration and retraining

The selected baseline has no fitted model to retrain. Revisit its thresholds and peer definitions only through a documented development-and-validation process, keeping an independent evaluation window separate.

Reconsider machine learning only if suitable independent labels and repeated validation show useful gains over the frozen baseline. A data-quality alert should prompt investigation, not automatic retraining or deployment.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

This notebook exports the historical ranked queue to work/outputs/content_action_queue.csv and a configuration-and-count receipt to work/outputs/action_playbook_receipt.json.

The queue CSV stays outside GitHub and is regenerated by running the notebook. The JSON receipt will be committed. It records this run’s settings and counts; it does not contain a new model-validation result.

The final paper should use the queue as a historical decision-support example and trace validation claims to the executed Week 5 and Week 6 notebooks. No human reviews, implemented changes, or traffic benefits are claimed.

In [9]:
import json
from datetime import datetime, timezone

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Export the queue privately within this runtime.
queue_path = output_dir / "content_action_queue.csv"
queue.to_csv(queue_path, index=False)

# Check the saved queue.
saved = pd.read_csv(queue_path)

assert len(saved) == len(queue)
assert saved["rank"].tolist() == queue["rank"].tolist()
assert np.allclose(saved["score"], queue["score"])
assert saved["review_status"].eq("NOT_REVIEWED").all()

action_counts = {
    str(action): int(count)
    for action, count in queue["action"].value_counts().items()
}

receipt = {
    "assignment": "ML-10",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": "FlyRank/internship-warehouse",
    "source_partition": "2026-03",
    "input_window": ["2026-03-01", "2026-03-20"],
    "use": "Historical decision-support demonstration",
    "selected_method": "Week 4 peer-gap baseline",
    "score_formula": (
        "impressions * max(peer_median_ctr - ctr, 0)"
    ),
    "minimum_observed_days": 10,
    "availability": "GSC and GA4 both TRUE",
    "minimum_impressions_for_scoring": 100,
    "minimum_peer_items": 20,
    "position_edges": [0, 3, 10, 20, "infinity"],
    "volume_groups": ["100-499", "500-1999", "2000+"],
    "queue_items": int(len(queue)),
    "positive_score_items": int((queue["score"] > 0).sum()),
    "action_counts": action_counts,
    "actual_page_reviews_completed": 0,
    "implemented_changes": 0,
    "human_review_precision": None,
    "update_benefit": None,
    "pilot_assumptions": {
        "batch_size": 10,
        "triage_minutes_per_item": 15,
        "target_actionable_items_per_batch": 7,
        "low_yield_batches_before_review": 2,
        "relative_eligible_count_alert": 0.20
    },
    "validation_sources": [
        "work/notebooks/w05_model.ipynb",
        "work/notebooks/w06_validation_audit.ipynb"
    ],
    "new_validation_metrics_computed": False,
    "queue_csv": str(queue_path),
    "automatic_content_changes_allowed": False,
    "june_data_used": False
}

receipt_path = output_dir / "action_playbook_receipt.json"
receipt_path.write_text(
    json.dumps(receipt, indent=2),
    encoding="utf-8"
)

print("Queue export verified:", queue_path)
print("JSON receipt written:", receipt_path)
print("Queue items:", len(saved))
display(saved["action"].value_counts().to_frame("item_count"))


Queue export verified: work/outputs/content_action_queue.csv
JSON receipt written: work/outputs/action_playbook_receipt.json
Queue items: 6607


,item_count
action,
MONITOR,3285
REVIEW_SEARCH_APPEARANCE_AND_INTENT,3280
COLLECT_MORE_DATA,42


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.